# Calling Card Peak Plot Generation

This notebook uses `calling_card_peak_pipeline.py` to load male and female Egr1 calling card peak BED files, annotate nearest genes, and generate a Venn diagram for nearest-gene peak overlaps.

In [ ]:
from pathlib import Path
from calling_card_peak_pipeline import (
    read_peak_bed,
    annotate_peaks,
    filter_by_distance,
    extract_unique_genes,
    plot_venn,
)

output_dir = Path('output/cc_peak_plots')
output_dir.mkdir(parents=True, exist_ok=True)

male_bed = Path('Egr1CC_peak_MaleEgr1_VS_MaleWT_MACC2_window1000_YchromFiltered_window300_p05.bed')
female_bed = Path('Egr1CC_peak_FemaleEgr1_VS_FemaleWT_MACC2_window1000_YchromFiltered_window300_p05.bed')
reference = 'mm10'
bedtools_path = '/ref/rmlab/software/pycallingcards/bin'

In [ ]:
male_df = read_peak_bed(male_bed)
female_df = read_peak_bed(female_bed)
print('Male peaks:', len(male_df))
print('Female peaks:', len(female_df))

In [ ]:
male_ann = annotate_peaks('Male_Egr1CC', male_df, reference, bedtools_path, output_dir)
female_ann = annotate_peaks('Female_Egr1CC', female_df, reference, bedtools_path, output_dir)

male_20kb = filter_by_distance(male_ann, 'Distance1', 20000)
female_20kb = filter_by_distance(female_ann, 'Distance1', 20000)
print('Male peaks within 20kb:', len(male_20kb))
print('Female peaks within 20kb:', len(female_20kb))

In [ ]:
male_genes1, male_genes2, _, *_ = extract_unique_genes(male_20kb, 'Male_Egr1CC', output_dir)
female_genes1, female_genes2, _, *_ = extract_unique_genes(female_20kb, 'Female_Egr1CC', output_dir)
print('Male nearest genes:', len(male_genes1))
print('Female nearest genes:', len(female_genes1))

In [ ]:
venn_path = output_dir / 'male_female_nearest_gene_venn.png'
plot_venn(male_genes1, female_genes1, ('Male Egr1CC nearest', 'Female Egr1CC nearest'), venn_path)
print('Saved Venn diagram to', venn_path)